In [1]:
# =============================================================================
# PROFESSIONAL SUICIDE ANALYTICS DASHBOARD
# Visualizing the Link Between Suicide Rates, Literacy & Unemployment in India
# =============================================================================

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, html, dcc, Input, Output, callback_context
import dash_bootstrap_components as dbc
from jupyter_dash import JupyterDash
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. ADVANCED DATA PROCESSING & MISSING VALUE HANDLING
# =============================================================================

def load_and_process_data(file_path):
    """Load and clean the dataset with comprehensive preprocessing and missing value handling"""
    df = pd.read_csv(file_path)
    
    print("Initial data shape:", df.shape)
    print("Missing values before cleaning:")
    print(df.isnull().sum())
    
    # Handle missing values strategically
    # For literacy rates - use median by region or forward fill
    literacy_cols = ["Literacy_Total", "Literacy_Male", "Literacy_Female"]
    for col in literacy_cols:
        df[col] = df[col].fillna(df[col].median())
    
    # For unemployment - use national average
    if df["Unemployment_Rate"].isnull().any():
        df["Unemployment_Rate"] = df["Unemployment_Rate"].fillna(df["Unemployment_Rate"].median())
    
    # For population - use forward fill or interpolation
    if df["Population"].isnull().any():
        df["Population"] = df["Population"].fillna(method='ffill')
    
    # For suicide rates - this is critical, use median of similar literacy states
    if df["Suicide_Rate_Per_Lakh"].isnull().any():
        df["Suicide_Rate_Per_Lakh"] = df["Suicide_Rate_Per_Lakh"].fillna(df["Suicide_Rate_Per_Lakh"].median())
    
    # Ensure numeric columns
    numeric_cols = ["Suicide_Rate_Per_Lakh", "Literacy_Total", "Literacy_Male", 
                   "Literacy_Female", "Unemployment_Rate", "Population"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Final cleanup - drop rows with critical missing data
    df = df.dropna(subset=["State", "Suicide_Rate_Per_Lakh"])
    df["State"] = df["State"].str.strip()
    
    # Feature engineering
    df["Literacy_Gap"] = df["Literacy_Male"] - df["Literacy_Female"]
    df["Risk_Category"] = pd.cut(df["Suicide_Rate_Per_Lakh"], 
                                bins=[0, 10, 20, 30, float('inf')], 
                                labels=["Low", "Medium", "High", "Critical"])
    
    # Additional derived features
    df["Education_Index"] = (df["Literacy_Total"] - df["Literacy_Total"].min()) / (df["Literacy_Total"].max() - df["Literacy_Total"].min())
    df["Economic_Stress"] = df["Unemployment_Rate"] * (1 - df["Education_Index"])
    
    print("Final data shape:", df.shape)
    print("Missing values after cleaning:")
    print(df.isnull().sum())
    
    return df

# Load data
file_path = r"C:\Users\Ashirwad\Downloads\merged_final_data_cleaned.csv"
df = load_and_process_data(file_path)

# =============================================================================
# 2. ADVANCED ANALYTICS & CORRELATION INSIGHTS
# =============================================================================

def compute_advanced_insights(df):
    """Compute comprehensive insights with correlation analysis and solutions"""
    insights = {}
    
    # Basic statistics
    insights['avg_suicide'] = df["Suicide_Rate_Per_Lakh"].mean()
    insights['median_suicide'] = df["Suicide_Rate_Per_Lakh"].median()
    insights['avg_literacy'] = df["Literacy_Total"].mean()
    insights['avg_unemployment'] = df["Unemployment_Rate"].mean()
    insights['total_population'] = df["Population"].sum()
    
    # Extreme states
    insights['highest_suicide'] = df.loc[df["Suicide_Rate_Per_Lakh"].idxmax()]
    insights['lowest_suicide'] = df.loc[df["Suicide_Rate_Per_Lakh"].idxmin()]
    insights['highest_literacy'] = df.loc[df["Literacy_Total"].idxmax()]
    insights['lowest_literacy'] = df.loc[df["Literacy_Total"].idxmin()]
    
    # Correlations
    corr_cols = ["Suicide_Rate_Per_Lakh", "Literacy_Total", "Unemployment_Rate", "Literacy_Gap"]
    insights['correlation_matrix'] = df[corr_cols].corr()
    
    # Advanced correlation insights with solutions
    literacy_corr = insights['correlation_matrix'].loc['Suicide_Rate_Per_Lakh', 'Literacy_Total']
    unemployment_corr = insights['correlation_matrix'].loc['Suicide_Rate_Per_Lakh', 'Unemployment_Rate']
    gender_gap_corr = insights['correlation_matrix'].loc['Suicide_Rate_Per_Lakh', 'Literacy_Gap']
    
    insights['correlation_insights'] = {
        'literacy_impact': {
            'correlation': literacy_corr,
            'direction': 'negative' if literacy_corr < 0 else 'positive',
            'strength': 'strong' if abs(literacy_corr) > 0.5 else 'moderate' if abs(literacy_corr) > 0.3 else 'weak',
            'solution': "Increase literacy programs" if literacy_corr < -0.2 else "Monitor literacy quality",
            'prediction': f"1% increase in literacy may lead to {abs(literacy_corr * 100):.1f}% change in suicide rates"
        },
        'unemployment_impact': {
            'correlation': unemployment_corr,
            'direction': 'positive' if unemployment_corr > 0 else 'negative',
            'strength': 'strong' if abs(unemployment_corr) > 0.5 else 'moderate' if abs(unemployment_corr) > 0.3 else 'weak',
            'solution': "Create job opportunities" if unemployment_corr > 0.2 else "Improve job quality",
            'prediction': f"1% increase in unemployment may lead to {abs(unemployment_corr * 100):.1f}% change in suicide rates"
        },
        'gender_gap_impact': {
            'correlation': gender_gap_corr,
            'direction': 'positive' if gender_gap_corr > 0 else 'negative',
            'strength': 'strong' if abs(gender_gap_corr) > 0.5 else 'moderate' if abs(gender_gap_corr) > 0.3 else 'weak',
            'solution': "Promote gender equality in education" if abs(gender_gap_corr) > 0.2 else "Maintain current policies"
        }
    }
    
    # Risk categorization
    insights['risk_distribution'] = df["Risk_Category"].value_counts()
    
    # Enhanced insights - cleaner and more concise
    insights['key_findings'] = {
        'literacy_impact': f"States with >80% literacy: {df[df['Literacy_Total'] > 80]['Suicide_Rate_Per_Lakh'].mean():.1f} vs {df[df['Literacy_Total'] <= 80]['Suicide_Rate_Per_Lakh'].mean():.1f} (lower literacy)",
        'unemployment_effect': f"High unemployment states (>15%): {df[df['Unemployment_Rate'] > 15]['Suicide_Rate_Per_Lakh'].mean():.1f} vs {df[df['Unemployment_Rate'] <= 15]['Suicide_Rate_Per_Lakh'].mean():.1f} (normal rates)",
        'gender_gap_effect': f"Large gender gap (>15%): {df[df['Literacy_Gap'] > 15]['Suicide_Rate_Per_Lakh'].mean():.1f} vs {df[df['Literacy_Gap'] <= 15]['Suicide_Rate_Per_Lakh'].mean():.1f} (smaller gaps)",
        'economic_stress': f"Economic stress correlation: {df['Economic_Stress'].corr(df['Suicide_Rate_Per_Lakh']):.3f}",
        'risk_states': f"{len(df[df['Risk_Category'] == 'Critical'])} Critical + {len(df[df['Risk_Category'] == 'High'])} High risk states",
        'weighted_national': f"Population-weighted national rate: {(df['Suicide_Rate_Per_Lakh'] * df['Population']).sum() / df['Population'].sum():.2f} per lakh"
    }
    
    return insights

insights = compute_advanced_insights(df)

# =============================================================================
# 3. ENHANCED VISUALIZATION COMPONENTS
# =============================================================================

# Professional color palette
COLORS = {
    'primary': '#1f4e79',
    'secondary': '#2d5aa0', 
    'success': '#28a745',
    'danger': '#dc3545',
    'warning': '#ffc107',
    'info': '#17a2b8',
    'light': '#f8f9fa',
    'dark': '#343a40',
    'accent': '#6f42c1',
    'gradient': ['#1f4e79', '#2d5aa0', '#28a745', '#dc3545']
}

def create_enhanced_scatter():
    """Enhanced scatter plot with trend analysis"""
    fig = px.scatter(
        df, 
        x="Literacy_Total", 
        y="Suicide_Rate_Per_Lakh",
        size="Population", 
        color="Economic_Stress",
        hover_name="State",
        hover_data={
            'Literacy_Total': ':.1f',
            'Suicide_Rate_Per_Lakh': ':.2f',
            'Unemployment_Rate': ':.1f',
            'Economic_Stress': ':.3f',
            'Population': ':,d'
        },
        labels={
            "Literacy_Total": "Literacy Rate (%)",
            "Suicide_Rate_Per_Lakh": "Suicide Rate (per lakh)",
            "Economic_Stress": "Economic Stress Index"
        },
        title="<b>Literacy vs Suicide Rate Analysis</b><br><sub>Size = Population | Color = Economic Stress | Trend Line Shows Correlation</sub>",
        color_continuous_scale="RdYlBu_r",
        template="plotly_white"
    )
    
    # Add trend line with equation
    fig.add_traces(px.scatter(df, x="Literacy_Total", y="Suicide_Rate_Per_Lakh", 
                             trendline="ols", trendline_color_override=COLORS['danger']).data[1:])
    
    fig.update_layout(
        height=480,
        font=dict(family="Inter, Arial", size=12),
        title_font_size=16,
        showlegend=True,
        legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        margin=dict(l=0, r=0, t=60, b=0)
    )
    
    return fig

def create_correlation_heatmap():
    """Enhanced correlation heatmap with insights"""
    corr_matrix = insights['correlation_matrix']
    
    # Create annotations with strength indicators
    annotations = []
    for i, row in enumerate(corr_matrix.index):
        for j, col in enumerate(corr_matrix.columns):
            value = corr_matrix.iloc[i, j]
            strength = "Strong" if abs(value) > 0.5 else "Moderate" if abs(value) > 0.3 else "Weak"
            annotations.append(
                dict(x=j, y=i, text=f"{value:.3f}<br>{strength}", 
                     showarrow=False, font=dict(color="white", size=10))
            )
    
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=['Suicide Rate', 'Literacy', 'Unemployment', 'Gender Gap'],
        y=['Suicide Rate', 'Literacy', 'Unemployment', 'Gender Gap'],
        colorscale='RdBu',
        zmid=0,
        showscale=True,
        colorbar=dict(title="Correlation Strength")
    ))
    
    fig.update_layout(
        annotations=annotations,
        title="<b>Correlation Matrix</b><br><sub>Understanding Variable Relationships</sub>",
        template="plotly_white",
        height=400,
        font=dict(family="Inter, Arial", size=11),
        title_font_size=16,
        margin=dict(l=0, r=0, t=60, b=0)
    )
    
    return fig

def create_comprehensive_ranking():
    """Comprehensive state ranking with multiple metrics"""
    top_states = df.nlargest(15, "Suicide_Rate_Per_Lakh")
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Top 15 States by Suicide Rate", "Literacy vs Unemployment Profile"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}]],
        column_widths=[0.6, 0.4]
    )
    
    # Bar chart for suicide rates
    fig.add_trace(
        go.Bar(
            x=top_states["Suicide_Rate_Per_Lakh"],
            y=top_states["State"],
            orientation='h',
            marker_color=COLORS['danger'],
            name="Suicide Rate",
            text=top_states["Suicide_Rate_Per_Lakh"].round(1),
            textposition='outside'
        ),
        row=1, col=1
    )
    
    # Literacy rates for top suicide states
    fig.add_trace(
        go.Scatter(
            x=top_states["Literacy_Total"],
            y=top_states["State"],
            mode='markers',
            marker=dict(size=12, color=COLORS['primary']),
            name="Literacy Rate",
        ),
        row=1, col=2
    )
    
    # Unemployment rates
    fig.add_trace(
        go.Scatter(
            x=top_states["Unemployment_Rate"],
            y=top_states["State"],
            mode='markers',
            marker=dict(size=12, color=COLORS['secondary'], symbol='square'),
            name="Unemployment Rate",
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        height=500,
        template="plotly_white",
        title_text="<b>Comprehensive State Analysis</b>",
        font=dict(family="Inter, Arial", size=10),
        showlegend=True
    )
    
    fig.update_xaxes(title_text="Suicide Rate (per lakh)", row=1, col=1)
    fig.update_xaxes(title_text="Rate (%)", row=1, col=2)
    
    return fig

def create_gender_literacy_analysis():
    """Gender literacy gap analysis"""
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Gender Literacy Gap vs Suicide Rate", "Male vs Female Literacy Rates"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Gap analysis
    fig.add_trace(
        go.Scatter(
            x=df["Literacy_Gap"],
            y=df["Suicide_Rate_Per_Lakh"],
            mode='markers',
            marker=dict(
                size=8,
                color=df["Unemployment_Rate"],
                colorscale='Viridis',
                colorbar=dict(title="Unemployment Rate"),
                line=dict(width=1, color='white')
            ),
            text=df["State"],
            hovertemplate="<b>%{text}</b><br>Gender Gap: %{x:.1f}%<br>Suicide Rate: %{y:.2f}<extra></extra>",
            name="States"
        ),
        row=1, col=1
    )
    
    # Male vs Female literacy
    fig.add_trace(
        go.Scatter(
            x=df["Literacy_Male"],
            y=df["Literacy_Female"],
            mode='markers',
            marker=dict(
                size=8,
                color=df["Suicide_Rate_Per_Lakh"],
                colorscale='Reds',
                line=dict(width=1, color='white')
            ),
            text=df["State"],
            hovertemplate="<b>%{text}</b><br>Male: %{x:.1f}%<br>Female: %{y:.1f}%<extra></extra>",
            name="Gender Comparison"
        ),
        row=1, col=2
    )
    
    # Add diagonal line for equal literacy
    fig.add_trace(
        go.Scatter(
            x=[40, 100],
            y=[40, 100],
            mode='lines',
            line=dict(dash='dash', color='gray'),
            name="Equal Literacy Line",
            showlegend=False
        ),
        row=1, col=2
    )
    
    fig.update_layout(
        height=400,
        template="plotly_white",
        title_text="<b>Gender Literacy Analysis</b>",
        font=dict(family="Inter, Arial", size=10)
    )
    
    fig.update_xaxes(title_text="Literacy Gap (Male - Female)", row=1, col=1)
    fig.update_xaxes(title_text="Male Literacy Rate (%)", row=1, col=2)
    fig.update_yaxes(title_text="Suicide Rate (per lakh)", row=1, col=1)
    fig.update_yaxes(title_text="Female Literacy Rate (%)", row=1, col=2)
    
    return fig

def create_risk_distribution():
    """Enhanced risk distribution with actionable insights"""
    risk_counts = insights['risk_distribution']
    
    fig = go.Figure(data=[go.Pie(
        labels=[f"{label}<br>({count} states)" for label, count in zip(risk_counts.index, risk_counts.values)],
        values=risk_counts.values,
        hole=0.5,
        marker_colors=[COLORS['success'], COLORS['warning'], COLORS['accent'], COLORS['danger']],
        textinfo='label+percent',
        textfont_size=11,
        pull=[0, 0, 0.1, 0.2]  # Emphasize high-risk categories
    )])
    
    fig.update_layout(
        title="<b>Risk Category Distribution</b><br><sub>States Classified by Suicide Rate Risk Levels</sub>",
        template="plotly_white",
        height=400,
        font=dict(family="Inter, Arial", size=11),
        title_font_size=14,
        showlegend=True,
        legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.05),
        margin=dict(l=0, r=0, t=60, b=0)
    )
    
    # Add center annotation
    fig.add_annotation(
        x=0.5, y=0.5,
        text=f"<b>29</b><br>Total States",
        showarrow=False,
        font=dict(size=16, color=COLORS['primary']),
        xref="paper", yref="paper"
    )
    
    return fig

def create_top_insights_chart():
    """Create additional insights visualization"""
    # Prepare data for multiple categories
    categories = ['High Literacy\n(>80%)', 'Low Literacy\n(<60%)', 'High Unemployment\n(>15%)', 'Large Gender Gap\n(>15%)', 'Small Gender Gap\n(<10%)']
    suicide_rates = [
        df[df['Literacy_Total'] > 80]['Suicide_Rate_Per_Lakh'].mean(),
        df[df['Literacy_Total'] < 60]['Suicide_Rate_Per_Lakh'].mean(),
        df[df['Unemployment_Rate'] > 15]['Suicide_Rate_Per_Lakh'].mean(),
        df[df['Literacy_Gap'] > 15]['Suicide_Rate_Per_Lakh'].mean(),
        df[df['Literacy_Gap'] < 10]['Suicide_Rate_Per_Lakh'].mean()
    ]
    
    colors = [COLORS['success'], COLORS['danger'], COLORS['danger'], COLORS['warning'], COLORS['success']]
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=categories,
        y=suicide_rates,
        marker_color=colors,
        text=[f"{rate:.1f}" for rate in suicide_rates],
        textposition='outside',
        name="Average Suicide Rate"
    ))
    
    # Add national average line
    fig.add_hline(y=insights['avg_suicide'], line_dash="dash", line_color="red",
                  annotation_text=f"National Average: {insights['avg_suicide']:.1f}")
    
    fig.update_layout(
        title="<b>Impact Analysis by Categories</b><br><sub>Suicide Rates Across Different Socio-Economic Segments</sub>",
        xaxis_title="Category",
        yaxis_title="Average Suicide Rate (per lakh)",
        template="plotly_white",
        height=400,
        font=dict(family="Inter, Arial", size=11),
        showlegend=False,
        margin=dict(l=0, r=0, t=60, b=0)
    )
    
    return fig

# Create all visualizations
scatter_fig = create_enhanced_scatter()
heatmap_fig = create_correlation_heatmap()
ranking_fig = create_comprehensive_ranking()
gender_fig = create_gender_literacy_analysis()
risk_fig = create_risk_distribution()
insights_chart = create_top_insights_chart()

# =============================================================================
# 4. PROFESSIONAL DASHBOARD LAYOUT
# =============================================================================

app = JupyterDash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP, dbc.icons.FONT_AWESOME])

# Enhanced styling
header_style = {
    'background': 'linear-gradient(135deg, #1f4e79 0%, #2d5aa0 100%)',
    'color': 'white',
    'padding': '30px 20px',
    'marginBottom': '25px',
    'borderRadius': '12px',
    'textAlign': 'center',
    'boxShadow': '0 8px 25px rgba(0,0,0,0.15)'
}

card_style = {
    'height': '100%',
    'boxShadow': '0 4px 12px rgba(0, 0, 0, 0.08)',
    'border': 'none',
    'borderRadius': '12px',
    'transition': 'transform 0.2s ease',
}

kpi_card_style = {
    'textAlign': 'center',
    'padding': '25px 15px',
    'borderRadius': '12px',
    'background': 'linear-gradient(135deg, #f8f9fa 0%, #ffffff 100%)',
    'border': '1px solid #e9ecef'
}

app.layout = dbc.Container([
         # Enhanced Header
    dbc.Row([
        dbc.Col([
            html.Div([
                html.H1("SUICIDE RATE ANALYTICS DASHBOARD", 
                       style={'fontSize': '2.8rem', 'fontWeight': '700', 'marginBottom': '12px', 'letterSpacing': '1px'}),
                html.H3("Visualizing the Link Between Suicide Rates, Literacy & Unemployment in India",
                       style={'fontSize': '1.3rem', 'fontWeight': '300', 'opacity': '0.9', 'marginBottom': '8px'}),
                html.P("Advanced statistical analysis with actionable insights for policy makers and researchers",
                       style={'fontSize': '1.1rem', 'marginTop': '15px', 'opacity': '0.8'})
            ], style=header_style)
        ], width=12)
    ]),
    # KPI Cards with Better Spacing
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.I(className="fas fa-chart-line fa-3x", style={'color': COLORS['primary'], 'marginBottom': '15px'}),
                    html.H2(f"{insights['avg_suicide']:.1f}", style={'color': COLORS['primary'], 'fontWeight': 'bold'}),
                    html.P("Average Suicide Rate", style={'color': '#666', 'fontSize': '1rem', 'fontWeight': '500'}),
                    html.Small("per lakh population", style={'color': '#999'})
                ], style=kpi_card_style)
            ], style=card_style)
        ], md=3),
        
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.I(className="fas fa-graduation-cap fa-3x", style={'color': COLORS['success'], 'marginBottom': '15px'}),
                    html.H2(f"{insights['avg_literacy']:.1f}%", style={'color': COLORS['success'], 'fontWeight': 'bold'}),
                    html.P("Average Literacy Rate", style={'color': '#666', 'fontSize': '1rem', 'fontWeight': '500'}),
                    html.Small("across all states", style={'color': '#999'})
                ], style=kpi_card_style)
            ], style=card_style)
        ], md=3),
        
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.I(className="fas fa-briefcase fa-3x", style={'color': COLORS['warning'], 'marginBottom': '15px'}),
                    html.H2(f"{insights['avg_unemployment']:.1f}%", style={'color': COLORS['warning'], 'fontWeight': 'bold'}),
                    html.P("Average Unemployment", style={'color': '#666', 'fontSize': '1rem', 'fontWeight': '500'}),
                    html.Small("rate nationwide", style={'color': '#999'})
                ], style=kpi_card_style)
            ], style=card_style)
        ], md=3),
        
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    html.I(className="fas fa-exclamation-triangle fa-3x", style={'color': COLORS['danger'], 'marginBottom': '15px'}),
                    html.H2(f"{insights['highest_suicide']['Suicide_Rate_Per_Lakh']:.1f}", 
                           style={'color': COLORS['danger'], 'fontWeight': 'bold'}),
                    html.P("Highest Risk State", style={'color': '#666', 'fontSize': '1rem', 'fontWeight': '500'}),
                    html.Small(f"{insights['highest_suicide']['State']}", style={'color': '#999'})
                ], style=kpi_card_style)
            ], style=card_style)
        ], md=3)
    ], className="mb-4"),
    
     # Strategic Insights Section (Moved to Top)
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H4("🎯 STRATEGIC INSIGHTS & CORRELATIONS", 
                           style={'marginBottom': '0', 'color': COLORS['primary'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dbc.Row([
                        dbc.Col([
                            html.H6("📊 Correlation Analysis", style={'color': COLORS['secondary'], 'fontWeight': '600'}),
                            html.Ul([
                                html.Li([
                                    html.Strong("Literacy Impact: "),
                                    f"{insights['correlation_insights']['literacy_impact']['strength'].title()} "
                                    f"{insights['correlation_insights']['literacy_impact']['direction']} correlation "
                                    f"({insights['correlation_insights']['literacy_impact']['correlation']:.3f})"
                                ]),
                                html.Li([
                                    html.Strong("Unemployment Effect: "),
                                    f"{insights['correlation_insights']['unemployment_impact']['strength'].title()} "
                                    f"{insights['correlation_insights']['unemployment_impact']['direction']} correlation "
                                    f"({insights['correlation_insights']['unemployment_impact']['correlation']:.3f})"
                                ]),
				html.Li([
                                    html.Strong("Gender Gap Influence: "),
                                    f"{insights['correlation_insights']['gender_gap_impact']['strength'].title()} impact "
                                    f"({insights['correlation_insights']['gender_gap_impact']['correlation']:.3f})"
                                ])
                            ], style={'fontSize': '0.95rem', 'lineHeight': '1.7'})
                        ], md=6),
                        
                        dbc.Col([
                            html.H6("💡 Recommended Solutions", style={'color': COLORS['success'], 'fontWeight': '600'}),
                            html.Ul([
                                html.Li([
                                    html.Strong("Literacy Programs: "),
                                    insights['correlation_insights']['literacy_impact']['solution']
                                ]),
                                html.Li([
                                    html.Strong("Employment Strategy: "),
                                    insights['correlation_insights']['unemployment_impact']['solution']
                                ]),
                                html.Li([
                                    html.Strong("Gender Equality: "),
                                    insights['correlation_insights']['gender_gap_impact']['solution']
                                ])
                            ], style={'fontSize': '0.95rem', 'lineHeight': '1.7'}),
				 html.Hr(),
                            html.H6("📈 Key Predictions", style={'color': COLORS['accent'], 'fontWeight': '600'}),
                            html.P([
                                insights['correlation_insights']['literacy_impact']['prediction']
                            ], style={'fontSize': '0.9rem', 'fontStyle': 'italic', 'color': '#666'})
                        ], md=6)
                    ])
                ])
            ], style=card_style, color="light", outline=True)
        ], width=12)
    ], className="mb-4"),
                         
    # Primary Analysis Section
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-chart-scatter", style={'marginRight': '10px'}),
                        "PRIMARY CORRELATION ANALYSIS"
                    ], style={'marginBottom': '0', 'color': COLORS['primary'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dcc.Graph(figure=scatter_fig, config={'displayModeBar': False})
                ])
            ], style=card_style)
        ], md=8),
        
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-th", style={'marginRight': '10px'}),
                        "CORRELATION MATRIX"
                    ], style={'marginBottom': '0', 'color': COLORS['secondary'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dcc.Graph(figure=heatmap_fig, config={'displayModeBar': False})
                ])
            ], style=card_style)
        ], md=4)
    ], className="mb-4"),
    
    # Analysis Section
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-exclamation-triangle", style={'marginRight': '10px'}),
                        "RISK DISTRIBUTION"
                    ], style={'marginBottom': '0', 'color': COLORS['accent'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dcc.Graph(figure=risk_fig, config={'displayModeBar': False})
                ])
            ], style=card_style)
        ], md=6),
        
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-chart-bar", style={'marginRight': '10px'}),
                        "IMPACT ANALYSIS"
                    ], style={'marginBottom': '0', 'color': COLORS['success'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dcc.Graph(figure=insights_chart, config={'displayModeBar': False})
                ])
            ], style=card_style)
        ], md=6)
    ], className="mb-4"),
    
    # Comprehensive Ranking
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-list-ol", style={'marginRight': '10px'}),
                        "COMPREHENSIVE STATE RANKING & ANALYSIS"
                    ], style={'marginBottom': '0', 'color': COLORS['primary'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dcc.Graph(figure=ranking_fig, config={'displayModeBar': False})
                ])
            ], style=card_style)
        ], width=12)
    ], className="mb-4"),
    
    # Gender Analysis
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-balance-scale", style={'marginRight': '10px'}),
                        "GENDER LITERACY GAP ANALYSIS"
                    ], style={'marginBottom': '0', 'color': COLORS['info'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dcc.Graph(figure=gender_fig, config={'displayModeBar': False})
                ])
            ], style=card_style)
        ], width=12)
    ], className="mb-4"),
    
    # IMPROVED Detailed Research Findings Section (Cleaner & Shorter)
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-microscope", style={'marginRight': '10px'}),
                        "RESEARCH FINDINGS"
                    ], style={'marginBottom': '0', 'color': COLORS['dark'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dbc.Row([
                        # Statistical Impact Column
                        dbc.Col([
                            html.H6([
                                html.I(className="fas fa-chart-pie", style={'marginRight': '8px', 'color': COLORS['primary']}),
                                "Statistical Impact"
                            ], style={'color': COLORS['primary'], 'marginBottom': '15px'}),
                            
                            # Clean stat cards
                            dbc.Alert([
                                html.Strong("High Literacy States: "),
                                insights['key_findings']['literacy_impact']
                            ], color="success", style={'fontSize': '0.9rem', 'marginBottom': '10px'}),
                            
                            dbc.Alert([
                                html.Strong("High Unemployment Impact: "),
                                insights['key_findings']['unemployment_effect']  
                            ], color="warning", style={'fontSize': '0.9rem', 'marginBottom': '10px'}),
                            
                            dbc.Alert([
                                html.Strong("Gender Gap Effect: "),
                                insights['key_findings']['gender_gap_effect']
                            ], color="info", style={'fontSize': '0.9rem', 'marginBottom': '0'})
                        ], md=6),
                        
                        # Policy Insights Column  
                        dbc.Col([
                            html.H6([
                                html.I(className="fas fa-flag", style={'marginRight': '8px', 'color': COLORS['danger']}),
                                "Policy Insights"
                            ], style={'color': COLORS['danger'], 'marginBottom': '15px'}),
                            
                            # Key metrics cards
                            dbc.Card([
                                dbc.CardBody([
                                    html.H6("Economic Stress Factor", style={'color': COLORS['accent'], 'fontSize': '0.9rem', 'marginBottom': '8px'}),
                                    html.P(insights['key_findings']['economic_stress'], 
                                          style={'fontSize': '0.85rem', 'marginBottom': '0'})
                                ])
                            ], style={'marginBottom': '10px', 'backgroundColor': '#f8f9fa'}),
                            
                            dbc.Card([
                                dbc.CardBody([
                                    html.H6("High-Risk States", style={'color': COLORS['danger'], 'fontSize': '0.9rem', 'marginBottom': '8px'}),
                                    html.P(insights['key_findings']['risk_states'], 
                                          style={'fontSize': '0.85rem', 'marginBottom': '0'})
                                ])
                            ], style={'marginBottom': '10px', 'backgroundColor': '#fff5f5'}),
                            
                            dbc.Card([
                                dbc.CardBody([
                                    html.H6("National Weighted Rate", style={'color': COLORS['info'], 'fontSize': '0.9rem', 'marginBottom': '8px'}),
                                    html.P(insights['key_findings']['weighted_national'], 
                                          style={'fontSize': '0.85rem', 'marginBottom': '0'})
                                ])
                            ], style={'marginBottom': '0', 'backgroundColor': '#f0f8ff'})
                        ], md=6)
                    ])
                ])
            ], style=card_style, color="light", outline=True)
        ], width=12)
    ], className="mb-4"),
    
    # Interactive State Explorer
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader([
                    html.H5([
                        html.I(className="fas fa-search", style={'marginRight': '10px'}),
                        "INTERACTIVE STATE EXPLORER"
                    ], style={'marginBottom': '0', 'color': COLORS['info'], 'fontWeight': '600'})
                ]),
                dbc.CardBody([
                    dbc.Row([
                        dbc.Col([
                            html.Label("Select State for Detailed Analysis:", 
                                     style={'fontWeight': '600', 'color': COLORS['dark']}),
                            dcc.Dropdown(
                                id="state-dropdown",
                                options=[{"label": state, "value": state} for state in sorted(df["State"].unique())],
                                placeholder="Choose a state for detailed analysis...",
                                style={'marginBottom': '25px'}
                            ),
                            html.Div(id="state-analysis")
                        ], md=6),
                        
                        dbc.Col([
                            html.Div(id="state-comparison")
                        ], md=6)
                    ])
                ])
            ], style=card_style)
        ], width=12)
    ], className="mb-5"),
    
     # Enhanced Footer
dbc.Row([
    dbc.Col([
        html.Hr(style={'margin': '40px 0 20px 0'}),
        
        # Developer Section
        dbc.Card([
            dbc.CardBody([
                dbc.Row([
                    # Developer Info Column
                    dbc.Col([
                        html.Div([
                            html.H6([
                                html.I(className="fas fa-chart-bar", style={'marginRight': '10px', 'color': COLORS['primary']}),
                                "Suicide Rate Analytics Dashboard"
                            ], style={'color': COLORS['primary'], 'fontWeight': '600', 'marginBottom': '15px'})
                        ])
                    ], md=6, style={'borderRight': '1px solid #e9ecef'}),
                    
                    # Dashboard Info Column
                    dbc.Col([
                        html.P([
                            html.I(className="fas fa-chart-bar", style={'marginRight': '8px'}),
                            "Data Source: Government of India Statistics",
                            html.Br(),
                            html.I(className="fas fa-flask", style={'marginRight': '8px', 'marginTop': '5px'}),
                            "Advanced Analytics: Statistical Correlations",
                            html.Br(),
                            html.I(className="fas fa-calendar", style={'marginRight': '8px', 'marginTop': '5px'}),
                            "Dashboard Version: 2024-Professional",
                            html.Br(),
                            html.I(className="fas fa-bolt", style={'marginRight': '8px', 'marginTop': '5px'}),
                            "Research-Grade Analysis"
                        ], style={'color': '#666', 'fontSize': '0.85rem', 'marginBottom': '0', 'lineHeight': '1.8'})
                    ], md=6)
                ])
            ], style={'padding': '20px 30px'})
        ], color="light", style={'borderRadius': '12px', 'boxShadow': '0 2px 8px rgba(0,0,0,0.08)'})
    ], width=12)
  ])
    
], fluid=True, style={'backgroundColor': '#f8f9fa', 'fontFamily': 'Inter, Arial, sans-serif', 'padding': '20px'})

# =============================================================================
# 5. ENHANCED INTERACTIVE CALLBACKS
# =============================================================================

@app.callback(
    [Output("state-analysis", "children"),
     Output("state-comparison", "children")],
    [Input("state-dropdown", "value")]
)
def update_state_explorer(selected_state):
    if not selected_state:
        return [
            dbc.Card([
                dbc.CardBody([
                    html.Div([
                        html.I(className="fas fa-search fa-3x", style={'color': '#ccc', 'marginBottom': '20px'}),
                        html.H5("Select a state to view comprehensive analysis", 
                               style={'color': '#666', 'fontWeight': '500'}),
                        html.P("Choose from the dropdown above to see detailed metrics, risk assessment, and comparative analysis.",
                               style={'color': '#888', 'fontSize': '0.9rem'})
                    ], style={'textAlign': 'center', 'padding': '50px 20px'})
                ])
            ], color="light", style=card_style)
        ], ""
    
    # Get state data
    state_data = df[df["State"] == selected_state].iloc[0]
    
    # Determine risk color
    risk_color = {
        'Low': COLORS['success'],
        'Medium': COLORS['warning'], 
        'High': COLORS['accent'],
        'Critical': COLORS['danger']
    }.get(state_data['Risk_Category'], COLORS['primary'])
    
    # State analysis card
    analysis = dbc.Card([
        dbc.CardHeader([
            html.H5([
                html.I(className="fas fa-map-marker-alt", style={'marginRight': '8px'}),
                f"{selected_state}"
            ], style={'marginBottom': '5px', 'color': COLORS['primary']}),
            html.P("Comprehensive State Profile", style={'marginBottom': '0', 'color': '#666', 'fontSize': '0.9rem'})
        ]),
        dbc.CardBody([
            # Risk and Suicide Rate
            dbc.Row([
                dbc.Col([
                    html.Div([
                        html.H6("Suicide Rate", style={'color': '#666', 'fontSize': '0.9rem', 'marginBottom': '5px'}),
                        html.H3(f"{state_data['Suicide_Rate_Per_Lakh']:.2f}", 
                               style={'color': COLORS['danger'], 'fontWeight': 'bold', 'marginBottom': '5px'}),
                        html.Small("per lakh population", style={'color': '#999'})
                    ], style={'textAlign': 'center', 'padding': '15px', 'backgroundColor': '#fff5f5', 'borderRadius': '8px'})
                ], md=6),
                dbc.Col([
                    html.Div([
                        html.H6("Risk Level", style={'color': '#666', 'fontSize': '0.9rem', 'marginBottom': '5px'}),
                        html.H3(f"{state_data['Risk_Category']}", 
                               style={'color': risk_color, 'fontWeight': 'bold', 'marginBottom': '5px'}),
                        html.Small("classification", style={'color': '#999'})
                    ], style={'textAlign': 'center', 'padding': '15px', 'backgroundColor': '#f8f9fa', 'borderRadius': '8px'})
                ], md=6)
            ], className="mb-3"),
            
            html.Hr(),
            
            # Detailed Metrics
            dbc.Row([
                dbc.Col([
                    html.H6([
                        html.I(className="fas fa-graduation-cap", style={'marginRight': '8px'}),
                        "Education Metrics"
                    ], style={'color': COLORS['success'], 'marginBottom': '10px'}),
                    html.P([
                        html.Strong("Total Literacy: "), f"{state_data['Literacy_Total']:.1f}%"
                    ], style={'marginBottom': '8px'}),
                    html.P([
                        html.Strong("Male Literacy: "), f"{state_data['Literacy_Male']:.1f}%"
                    ], style={'marginBottom': '8px'}),
                    html.P([
                        html.Strong("Female Literacy: "), f"{state_data['Literacy_Female']:.1f}%"
                    ], style={'marginBottom': '8px'}),
                    html.P([
                        html.Strong("Gender Gap: "), f"{state_data['Literacy_Gap']:.1f}%"
                    ], style={'marginBottom': '0', 'color': COLORS['warning'] if state_data['Literacy_Gap'] > 10 else COLORS['success']})
                ], md=6),
                dbc.Col([
                    html.H6([
                        html.I(className="fas fa-chart-line", style={'marginRight': '8px'}),
                        "Economic & Demographics"
                    ], style={'color': COLORS['info'], 'marginBottom': '10px'}),
                    html.P([
                        html.Strong("Unemployment Rate: "), f"{state_data['Unemployment_Rate']:.1f}%"
                    ], style={'marginBottom': '8px'}),
                    html.P([
                        html.Strong("Population: "), f"{int(state_data['Population']):,}"
                    ], style={'marginBottom': '8px'}),
                    html.P([
                        html.Strong("Economic Stress: "), f"{state_data['Economic_Stress']:.3f}"
                    ], style={'marginBottom': '8px'}),
                    html.P([
                        html.Strong("Education Index: "), f"{state_data['Education_Index']:.3f}"
                    ], style={'marginBottom': '0'})
                ], md=6)
            ])
        ])
    ], color="light", style=card_style)
    
    # Comparison with national averages
    comparison = dbc.Card([
        dbc.CardHeader([
            html.H5([
                html.I(className="fas fa-chart-bar", style={'marginRight': '8px'}),
                "National Comparison"
            ], style={'marginBottom': '5px', 'color': COLORS['info']}),
            html.P("Performance vs National Averages", style={'marginBottom': '0', 'color': '#666', 'fontSize': '0.9rem'})
        ]),
        dbc.CardBody([
            # Comparison metrics
            html.Div([
                # Suicide Rate Comparison
                html.Div([
                    html.P([
                        html.Strong("Suicide Rate: "),
                        html.Span(
                            f"{((state_data['Suicide_Rate_Per_Lakh'] / insights['avg_suicide'] - 1) * 100):+.1f}%",
                            style={
                                'color': COLORS['danger'] if state_data['Suicide_Rate_Per_Lakh'] > insights['avg_suicide'] else COLORS['success'],
                                'fontWeight': 'bold',
                                'fontSize': '1.1rem'
                            }
                        ),
                        html.Br(),
                        html.Small(" vs national average", style={'color': '#666'})
                    ], style={'marginBottom': '12px', 'padding': '10px', 'backgroundColor': '#f8f9fa', 'borderRadius': '6px'})
                ]),
                
                # Literacy Comparison
                html.Div([
                    html.P([
                        html.Strong("Literacy Rate: "),
                        html.Span(
                            f"{((state_data['Literacy_Total'] / insights['avg_literacy'] - 1) * 100):+.1f}%",
                            style={
                                'color': COLORS['success'] if state_data['Literacy_Total'] > insights['avg_literacy'] else COLORS['danger'],
                                'fontWeight': 'bold',
                                'fontSize': '1.1rem'
                            }
                        ),
                        html.Br(),
                        html.Small(" vs national average", style={'color': '#666'})
                    ], style={'marginBottom': '12px', 'padding': '10px', 'backgroundColor': '#f8f9fa', 'borderRadius': '6px'})
                ]),
                
                # Unemployment Comparison
                html.Div([
                    html.P([
                        html.Strong("Unemployment: "),
                        html.Span(
                            f"{((state_data['Unemployment_Rate'] / insights['avg_unemployment'] - 1) * 100):+.1f}%",
                            style={
                                'color': COLORS['danger'] if state_data['Unemployment_Rate'] > insights['avg_unemployment'] else COLORS['success'],
                                'fontWeight': 'bold',
                                'fontSize': '1.1rem'
                            }
                        ),
                        html.Br(),
                        html.Small(" vs national average", style={'color': '#666'})
                    ], style={'marginBottom': '12px', 'padding': '10px', 'backgroundColor': '#f8f9fa', 'borderRadius': '6px'})
                ]),
                
                html.Hr(),
                
                # Recommendations
                html.Div([
                    html.H6([
                        html.I(className="fas fa-lightbulb", style={'marginRight': '8px'}),
                        "Some preventive measures"
                    ], style={'color': COLORS['accent'], 'marginBottom': '10px'}),
                    html.Ul([
                        html.Li("Focus on literacy programs" if state_data['Literacy_Total'] < insights['avg_literacy'] else "Maintain literacy standards"),
                        html.Li("Job creation initiatives" if state_data['Unemployment_Rate'] > insights['avg_unemployment'] else "Economic stability measures"),
                        html.Li("Gender equality programs" if state_data['Literacy_Gap'] > 10 else "Continue gender parity efforts"),
                        html.Li("Mental health support systems" if state_data['Risk_Category'] in ['High', 'Critical'] else "Preventive mental health programs")
                    ], style={'fontSize': '0.85rem', 'marginBottom': '0'})
                ])
            ])
        ])
    ], color="info", outline=True, style=card_style)
    
    return analysis, comparison

# =============================================================================
# 6. RUN THE DASHBOARD
# =============================================================================

if __name__ == "__main__":
    print("🚀 Launching Advanced Suicide Analytics Dashboard...")
    print("📊 Dashboard Features:")
  
    print("📍 Access URL: http://127.0.0.1:8055/")
    print("🎯 Ready for Policy Analysis and Decision Making!")
    
    app.run(mode="inline", port=8055, debug=True)

Initial data shape: (38, 8)
Missing values before cleaning:
State                    0
Literacy_Total           3
Literacy_Male            3
Literacy_Female          3
Unemployment_Rate        0
Total_Suicides           0
Population               0
Suicide_Rate_Per_Lakh    0
dtype: int64
Final data shape: (38, 12)
Missing values after cleaning:
State                    0
Literacy_Total           0
Literacy_Male            0
Literacy_Female          0
Unemployment_Rate        0
Total_Suicides           0
Population               0
Suicide_Rate_Per_Lakh    0
Literacy_Gap             0
Risk_Category            9
Education_Index          0
Economic_Stress          0
dtype: int64
🚀 Launching Advanced Suicide Analytics Dashboard...
📊 Dashboard Features:
📍 Access URL: http://127.0.0.1:8055/
🎯 Ready for Policy Analysis and Decision Making!
